# 04 — Representation learning: one-hot, embeddings, Word2Vec, GloVe-style co-occurrence

## Mental model

Text must become numbers before models can use it.

- One-hot vectors are sparse and do not encode meaning.
- Word embeddings are dense vectors where similar contexts produce similar vectors.
- Word2Vec learns by prediction over local windows.
- GloVe-style methods use global co-occurrence counts.

Founder lens: embeddings power semantic search, clustering, recommendations, duplicate detection, and retrieval.

In [ ]:
from pathlib import Path
import random
import numpy as np

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
DATA_DIR = ROOT / "data"
print("Project root:", ROOT)

In [ ]:
import re
import numpy as np
import pandas as pd
from collections import Counter, defaultdict
from sklearn.metrics.pairwise import cosine_similarity

df = pd.read_csv(DATA_DIR / "sample_customer_tickets.csv")
texts = df["text"].tolist()

def tokenize(text):
    return re.findall(r"[a-z]+", text.lower())

tokenized = [tokenize(t) for t in texts]
tokenized[:2]

## One-hot vectors

One-hot is simple but treats all words as equally unrelated.

In [ ]:
vocab = sorted({w for doc in tokenized for w in doc})
word_to_idx = {w: i for i, w in enumerate(vocab)}

def one_hot(word):
    v = np.zeros(len(vocab))
    if word in word_to_idx:
        v[word_to_idx[word]] = 1
    return v

for a, b in [("app", "dashboard"), ("app", "charged")]:
    sim = cosine_similarity([one_hot(a)], [one_hot(b)])[0, 0]
    print(a, b, sim)

## GloVe-style co-occurrence matrix + SVD

This is not full GloVe training, but it demonstrates the global co-occurrence idea: words that occur near similar words get similar vectors.

In [ ]:
window = 2
cooc = np.zeros((len(vocab), len(vocab)), dtype=float)
for doc in tokenized:
    for i, w in enumerate(doc):
        wi = word_to_idx[w]
        start = max(0, i - window)
        end = min(len(doc), i + window + 1)
        for j in range(start, end):
            if i == j:
                continue
            cj = word_to_idx[doc[j]]
            cooc[wi, cj] += 1.0 / abs(i - j)

# Positive PMI weighting
row_sum = cooc.sum(axis=1, keepdims=True) + 1e-9
col_sum = cooc.sum(axis=0, keepdims=True) + 1e-9
total = cooc.sum() + 1e-9
pmi = np.log((cooc * total + 1e-9) / (row_sum @ col_sum + 1e-9))
ppmi = np.maximum(pmi, 0)

# Low-rank embeddings by SVD
U, S, Vt = np.linalg.svd(ppmi, full_matrices=False)
embeddings = U[:, :20] * np.sqrt(S[:20])

def nearest(word, topn=8):
    if word not in word_to_idx:
        return []
    vec = embeddings[word_to_idx[word]].reshape(1, -1)
    sims = cosine_similarity(vec, embeddings)[0]
    order = np.argsort(sims)[::-1]
    return [(vocab[i], float(sims[i])) for i in order[1:topn+1]]

for word in ["app", "support", "billing", "model"]:
    print("\n", word)
    print(nearest(word, topn=5))

## Word2Vec with gensim

Word2Vec is self-supervised: the raw text creates training pairs. Skip-gram predicts context from center words; CBOW predicts center from context.

In [ ]:
try:
    from gensim.models import Word2Vec
    w2v = Word2Vec(
        sentences=tokenized,
        vector_size=50,
        window=3,
        min_count=1,
        sg=1,       # 1 = skip-gram, 0 = CBOW
        negative=5,
        epochs=200,
        seed=SEED,
        workers=1,
    )
    for word in ["app", "support", "billing", "model"]:
        print("\n", word)
        print(w2v.wv.most_similar(word, topn=5))
except Exception as e:
    print("gensim Word2Vec unavailable:", e)

## Document embeddings by averaging word vectors

This is a simple baseline for semantic search. Modern systems usually use contextual sentence embeddings, but averaging is useful for understanding and quick baselines.

In [ ]:
def doc_vector(tokens, keyed_vectors=None):
    if keyed_vectors is not None:
        vecs = [keyed_vectors[w] for w in tokens if w in keyed_vectors]
        if vecs:
            return np.mean(vecs, axis=0)
        return np.zeros(keyed_vectors.vector_size)
    # fallback to SVD embeddings
    vecs = [embeddings[word_to_idx[w]] for w in tokens if w in word_to_idx]
    return np.mean(vecs, axis=0) if vecs else np.zeros(embeddings.shape[1])

try:
    doc_embs = np.vstack([doc_vector(toks, w2v.wv) for toks in tokenized])
except NameError:
    doc_embs = np.vstack([doc_vector(toks) for toks in tokenized])

query = "billing problem charged twice"
q = doc_vector(tokenize(query), w2v.wv if 'w2v' in globals() else None).reshape(1, -1)
sims = cosine_similarity(q, doc_embs)[0]
for i in np.argsort(sims)[::-1][:5]:
    print(round(float(sims[i]), 3), df.loc[i, "text"])

## Limitations and product note

Classic word embeddings assign one vector per word type, so they struggle with multiple meanings:

```text
bank = river bank
bank = financial bank
```

Contextual embeddings from Transformers solve this better because a word's vector depends on the sentence.